# 1. Italy's public debt: history, arithmetic and the choices that shaped it

**Questions.** How did Italy's public debt reach roughly 135-155% of GDP? Which part of the increase came from primary deficits
(spending above revenues, excluding interest), which from the gap between the interest rate on the debt and nominal GDP growth
(the "snowball"), and which from one-off operations? What would the debt ratio be today under different past choices?

**Method.** The debt ratio obeys an accounting identity (European Commission, *Debt Sustainability Monitor*):

$$ d_t - d_{t-1} = \underbrace{-pb_t}_{\text{primary deficit}} + \underbrace{d_{t-1}\frac{i_t - \gamma_t}{1+\gamma_t}}_{\text{snowball}} + \underbrace{sfa_t}_{\text{stock-flow adjustment}} $$

with $i$ the effective interest rate on the debt, $\gamma$ nominal GDP growth (real growth $g$ and GDP deflator inflation $\pi$). The snowball is
split into an interest effect, a real-growth effect and an inflation effect. Counterfactuals replay the identity with changed components
while keeping the observed stock-flow adjustments.

**Data.** Eurostat government deficit and debt (excessive deficit procedure, `gov_10dd_edpt1`) and national accounts (`nama_10_gdp`),
general government (S13), from 1995; IMF World Economic Outlook debt ratio from 1980 (DataMapper); optionally the Bank of Italy's
historical series since 1861 saved as `data/external/debt_1861.csv` (columns `year,debt_pct`).

**Reading guide.** Figures in the markdown cells that refer to the period before 1995 are approximate values from the cited literature;
everything from 1995 onwards is computed in the code cells from official data.

In [ ]:
import os
import sys
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

ROOT = next(p for p in (Path.cwd(), *Path.cwd().parents) if (p / "scripts" / "public_debt").is_dir())
try:
    import public_debt
except ImportError:  # not installed: use the source folder (the extension must be built in place)
    sys.path.insert(0, str(ROOT / "scripts" / "public_debt"))
    import public_debt
from public_debt import accounting, data, dsa, fiscal, growth, synthetic
from public_debt.var import fit_var1

core = public_debt.require_cpp()

DATA_MODE = os.environ.get("PUBLIC_DEBT_DATA_MODE", "official")   # "official" (Eurostat, IMF) or "synthetic" (offline)
OFFICIAL = DATA_MODE == "official"
OUTPUT_DIR = ROOT / "outputs" / "debt_history"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
pd.set_option("display.float_format", lambda x: f"{x:,.2f}")
plt.rcParams.update({"figure.dpi": 110, "axes.grid": True, "grid.alpha": 0.3})
print(f"public_debt {public_debt.__version__} | data mode: {DATA_MODE}")
if not OFFICIAL:
    print("WARNING: synthetic data. Tables and charts are placeholders, not statistics about Italy.")

## 1.1 Data

In [ ]:
if OFFICIAL:
    it = data.fiscal_dataset("IT")
    peer = data.fiscal_dataset("EU27_2020")
else:
    it, peer = synthetic.fiscal_dataset(seed=5), synthetic.fiscal_dataset(seed=15)
print("Source:", it.attrs["source"])
print(f"Italy: {it.index.min()}-{it.index.max()}; debt {100 * it['debt'].iloc[0]:.1f}% -> {100 * it['debt'].iloc[-1]:.1f}% of GDP")

long_run = {}
if OFFICIAL:
    try:
        long_run["IMF World Economic Outlook"] = data.imf_series("GGXWDG_NGDP", "ITA")
    except Exception as exc:
        print("IMF series not available:", type(exc).__name__, exc)
historical = data.historical_debt()
if historical is not None:
    long_run["historical series (user file)"] = historical
display((100 * it[["debt", "balance", "primary_balance", "interest", "real_growth", "inflation", "effective_rate"]]).tail(8).round(2))

## 1.2 The long view (1861 to today)

| Period | Debt ratio (approx.) | What happened |
| --- | --- | --- |
| 1861-1913 | above 100% in the 1890s, lower before 1914 | The new Kingdom inherits pre-unitary debts and finances wars and railways; the Destra storica reaches a balanced budget in 1876; growth and primary surpluses reduce the ratio before the First World War. |
| 1914-1945 | war peaks, then erosion | War finance raises debt; inflation and the 1926 forced conversion of short-term debt reduce it; the Second World War and the 1943-1947 inflation wipe out most of its real value. |
| 1946-1969 | about 30-40% | The "economic miracle": fast growth keeps the ratio low. |
| 1970-1980 | about 40% to about 57% | Rapid expansion of social spending (earnings-related pensions, early-retirement rules for public employees, regional governments from 1970, national health service in 1978) without matching revenues; large deficits partly financed by the central bank; high inflation and negative real interest rates contain the ratio. |
| 1981-1994 | about 57% to about 120% | The 1981 "divorce" between the Treasury and the Bank of Italy ends monetary financing; disinflation brings real interest rates well above growth, but primary deficits continue until the early 1990s: interest spending reaches double digits as a share of GDP and the snowball dominates. |
| 1992-2000 | peak, then decline | The 1992 currency crisis forces emergency budgets, pension reforms (1992, 1995), large privatisations and primary surpluses of several points of GDP; convergence to the euro lowers interest rates. |
| 2001-2007 | slow decline to about 100-105% | The "euro dividend" of lower interest spending is only partly saved: the primary surplus erodes while growth stays weak. |
| 2008-2014 | to about 135% | Global financial crisis and sovereign debt crisis (spread above 500 basis points in November 2011); consolidation during a double-dip recession; falling nominal GDP makes the snowball positive despite primary surpluses. |
| 2015-2019 | stable around 135% | ECB quantitative easing lowers rates; growth remains low. |
| 2020-today | about 155% in 2020, then down | Pandemic deficits; strong nominal growth in 2021-2022 lowers the ratio, while building-renovation tax credits (Superbonus) keep deficits high until 2023. |

Sources: Francese and Pace (2008); Balassone, Francese and Pace (2013); Bank of Italy and ISTAT statistics; Cottarelli (2016).

In [ ]:
fig, ax = plt.subplots(figsize=(12, 4.5))
for label, series in long_run.items():
    ax.plot(series.index, series.values, lw=1.2, label=label)
ax.plot(it.index, 100 * it["debt"], "k-", lw=2, label="Eurostat (EDP), Italy")
ax.plot(peer.index, 100 * peer["debt"], "k--", lw=1, label="Eurostat, EU27")
ax.set(title="General government gross debt, % of GDP", ylabel="% of GDP")
ax.legend(fontsize=8)
fig.tight_layout()
fig.savefig(OUTPUT_DIR / "debt_long_run.png", dpi=150)

## 1.3 The key relationship: interest rate versus nominal growth

When the effective interest rate $i$ exceeds nominal growth $\gamma$, the debt ratio rises unless the primary balance is large enough:
the **debt-stabilising primary balance** is $d\,(i-\gamma)/(1+\gamma)$. The chart compares the actual primary balance with this
threshold year by year.

In [ ]:
stab = accounting.stabilising_primary_balance(it["debt"].shift(1), it["effective_rate"], it["real_growth"], it["inflation"])
fig, axes = plt.subplots(1, 2, figsize=(13, 4))
axes[0].plot(it.index, 100 * it["effective_rate"], label="effective interest rate i")
axes[0].plot(it.index, 100 * it["nominal_growth"], label="nominal GDP growth")
axes[0].axhline(0, color="k", lw=0.8)
axes[0].set(title="Interest rate versus growth (%)")
axes[0].legend(fontsize=8)
axes[1].bar(it.index, 100 * it["primary_balance"], color="C2", alpha=0.6, label="primary balance")
axes[1].plot(it.index, 100 * stab, "k.-", label="debt-stabilising primary balance")
axes[1].set(title="Primary balance and the stabilising threshold (% of GDP)")
axes[1].legend(fontsize=8)
fig.tight_layout()
fig.savefig(OUTPUT_DIR / "i_minus_g.png", dpi=150)
gap = (it["primary_balance"] - stab).dropna()
print(f"Years in which the primary balance exceeded the stabilising level: {(gap > 0).sum()} of {len(gap)}")

## 1.4 Decomposition of the change in the debt ratio

In [ ]:
dec = accounting.debt_decomposition(it)
comps = ["primary deficit", "interest", "real growth", "inflation", "stock-flow adjustment"]
fig, ax = plt.subplots(figsize=(12, 4.5))
bottom_pos = np.zeros(len(dec)); bottom_neg = np.zeros(len(dec))
for c in comps:
    vals = 100 * dec[c].to_numpy()
    ax.bar(dec.index, np.where(vals > 0, vals, 0), bottom=bottom_pos, label=c)
    ax.bar(dec.index, np.where(vals < 0, vals, 0), bottom=bottom_neg, color=ax.patches[-1].get_facecolor())
    bottom_pos += np.where(vals > 0, vals, 0); bottom_neg += np.where(vals < 0, vals, 0)
ax.plot(dec.index, 100 * dec["change in debt"], "k.-", label="change in debt ratio")
ax.axhline(0, color="k", lw=0.8)
ax.set(title="Contributions to the annual change in the debt ratio (percentage points of GDP)")
ax.legend(ncol=3, fontsize=8)
fig.tight_layout()
fig.savefig(OUTPUT_DIR / "decomposition.png", dpi=150)

last = int(dec.index.max())
periods = {"1996-1999 convergence to the euro": (1996, 1999), "2000-2007 early euro years": (2000, 2007),
           "2008-2013 financial and sovereign crises": (2008, 2013), "2014-2019 low rates, low growth": (2014, 2019),
           "2020-2021 pandemic": (2020, 2021), f"2022-{last} inflation and tax credits": (2022, last)}
periods = {k: v for k, v in periods.items() if v[0] <= last}
table = 100 * accounting.by_period(dec, periods)
display(table[["change in debt"] + comps].round(1))
table.to_csv(OUTPUT_DIR / "decomposition_by_period.csv")

## 1.5 Counterfactuals: what if different choices had been made?

Each counterfactual replays the identity from a starting year, changing one component and keeping everything else, including the
stock-flow adjustments, as observed. They ignore feedback effects (for example, higher primary surpluses would have lowered growth in
the short run and interest rates in the long run), so they measure orders of magnitude, not precise alternative histories.

1. **Saving the euro dividend.** The primary balance kept from 2001 at its 1997-2000 average instead of eroding.
2. **A lower risk premium.** An effective interest rate one percentage point lower from 2012 (after the sovereign crisis).
3. **Growing like the EU.** Italy's real growth replaced by the EU27 average from 1999.
4. **No pandemic-era tax credits and emergency deficits.** The primary balance of 2020-2023 capped at its 2019 level (an upper bound
   for the combined effect of emergency measures and building tax credits; the pandemic response was largely unavoidable).

In [ ]:
first, last = int(it.index.min()), int(it.index.max())
avg_9700 = it.loc[1997:2000, "primary_balance"].mean() if first <= 1997 else it["primary_balance"].iloc[:4].mean()
cf = {"observed": it["debt"]}
if first <= 2000:
    pb = it["primary_balance"].copy(); pb.loc[2001:] = np.maximum(pb.loc[2001:], avg_9700)
    cf["1. euro dividend saved"] = accounting.counterfactual(it, 2000, primary_balance=pb)
if first <= 2011 < last:
    cf["2. rate 1 pp lower from 2012"] = accounting.counterfactual(it, 2011, effective_rate=it["effective_rate"] - 0.01)
if first <= 1998:
    g = it["real_growth"].copy(); g.loc[1999:] = peer["real_growth"].reindex(g.loc[1999:].index)
    cf["3. EU27 real growth from 1999"] = accounting.counterfactual(it, 1998, real_growth=g)
if first <= 2019 and last >= 2020:
    pb = it["primary_balance"].copy(); pb.loc[2020:2023] = np.maximum(pb.loc[2020:2023], pb.loc[2019])
    cf["4. 2020-2023 primary balance at 2019 level"] = accounting.counterfactual(it, 2019, primary_balance=pb)
paths = pd.DataFrame(cf)
fig, ax = plt.subplots(figsize=(12, 4.5))
for c in paths:
    ax.plot(paths.index, 100 * paths[c], lw=2.5 if c == "observed" else 1.2, label=c, color="k" if c == "observed" else None)
ax.set(title="Debt ratio: observed and counterfactual paths (% of GDP)")
ax.legend(fontsize=8)
fig.tight_layout()
fig.savefig(OUTPUT_DIR / "counterfactuals.png", dpi=150)
summary = pd.DataFrame({"debt in last year (%)": 100 * paths.iloc[-1],
                        "difference vs observed (pp)": 100 * (paths.iloc[-1] - paths["observed"].iloc[-1])})
display(summary.round(1))

## 1.6 Lessons

What the arithmetic and the historical record indicate (the numbers above quantify them for the period since 1995):

1. **The debt was built mainly in the 1980s**, when primary deficits persisted after the end of monetary financing in 1981 and real
   interest rates exceeded growth. Fiscal policy took about a decade to adapt to the new monetary regime: the most costly delay in the
   country's fiscal history.
2. **Commitments with long-lasting costs were made without financing**: earnings-related pensions with generous early-retirement rules,
   indexation mechanisms and the expansion of public employment in the 1970s raised structural spending; pension reforms since 1992
   (especially the 1995 notional defined-contribution system) corrected the trend only gradually.
3. **Consolidation works when it is sustained and supported by growth and falling rates**, as in 1993-2000; it was not sustained in the
   2000s, when the lower interest bill was spent rather than saved.
4. **The snowball, not the primary balance, has driven most recent increases**: since the 1990s Italy has run primary surpluses in most
   years, yet weak nominal growth and crises (2008-2013, 2020) raised the ratio. Growth is the missing ingredient.
5. **Temporary measures can have permanent costs**: large tax credits with weak controls and high deadweight (Superbonus) added sizeable
   deficits in 2021-2023 at the time when inflation was lowering the ratio.
6. **Stock-flow adjustments matter**: privatisations in the 1990s reduced debt, while accumulation of financial assets, derivatives and
   statistical reclassifications raised it at times; they are the first place to look when the debt moves more than deficits suggest.

Notebook 2 examines where spending and revenues differ from other EU countries, notebook 3 the growth drivers, and notebook 4 the options
to put the ratio on a declining path.

### References

- Balassone, F., Francese, M. and Pace, A. (2013). Public debt and economic growth: Italy's first 150 years. In *The Oxford Handbook of the Italian Economy since Unification*. Oxford University Press.
- Cottarelli, C. (2016). *Il macigno. Perché il debito pubblico ci schiaccia e come si fa a liberarsene*. Feltrinelli.
- European Commission. *Debt Sustainability Monitor* (annual), methodology of the debt decomposition.
- Francese, M. and Pace, A. (2008). Il debito pubblico italiano dall'Unità a oggi. Una ricostruzione della serie storica. Banca d'Italia, Questioni di Economia e Finanza 31.
- Eurostat, `gov_10dd_edpt1` and `nama_10_gdp`; IMF, World Economic Outlook database.